In [4]:
from matplotlib.colors import ListedColormap
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
from itertools import product
import groupings as gp
from numpy import nan
import numpy as np
import pandas as pd
import warnings
import os
idx = pd.IndexSlice
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

work_path = r"C:/Users/damie/OneDrive/UW/research/dargan/projects/energy_emissions/" # base path for all work

raw_path = work_path + r"data/raw/" # path for saving raw, unprocessed data
processed_path = work_path + r"data/processed/" # path for saving processed data
viz_path = work_path + r"data/visualization/" # path for saving visualizations of the data

# data year selections
data_range = np.arange(1960, 2023, dtype=int) # select the years to be used in the processed data
data_range_emissions = np.arange(1971, 2023, dtype=int) # emissions data has a shorter range of years

# World Energy Balances

In order for the following code to run you must first download the following raw data sources to `raw_path`. Note that some of the links may not be current, so you may want to navigate to the most updated version.

- Population by Select Age Groups - Both Sexes: https://population.un.org/wpp/downloads?folder=Standard%20Projections&group=Population
	- Rename this file `population` (xlsx extension)
- Maddison Project Database: https://www.rug.nl/ggdc/historicaldevelopment/maddison/releases/maddison-project-database-2023
  - Rename this file `gdp` (xlsx extension)
- Summary Energy Balances: https://www.iea.org/data-and-statistics/data-product/world-energy-balances#energy-balances
  - Rename this file `energy` (txt extension)
- IEA-EDGAR CO2: https://edgar.jrc.ec.europa.eu/dataset_ghg2025
	- Rename this file `emissions` (xlsx extension)

## Raw Data Processing

### Loading raw data and reformatting

In [ ]:
# data scalings
population_scaling = 1e3 # converts to individuals
gdp_scaling = 1.47 # converts to 2024 US Dollars
energy_scaling = 1e6 # converts to megajoules
emissions_scaling = 1e9 # converts to grams

#### Population 

Raw data units are thousands of individuals

In [23]:
df = pd.read_excel(raw_path + "population.xlsx",
                    sheet_name = "Estimates",
                    header = 16)
population = df[["Region, subregion, country or area *",
                 "ISO3 Alpha-code",
                 "Year",
                 "Total"]]
population_ISO3 = population.pivot_table(index = "Year",
                                         columns = "ISO3 Alpha-code",
                                         values = "Total",
                                         aggfunc= "sum") # select individual countries
population_region = population.pivot_table(index = "Year",
                                           columns = "Region, subregion, country or area *",
                                           values = "Total",
                                           aggfunc="sum")
population_region = population_region[["Africa",
                                       "South-Eastern Asia"]] # select needed regions
population = pd.concat([population_ISO3, population_region],
                       axis=1)
population.index = population.index.astype(int)
population = population.loc[data_range, :] # select desired years
population.index.name = None
population = population_scaling * population
population.to_csv(processed_path + "population.csv")

#### GDP

Raw data units are 2011 US dollars

In [24]:
gdp_per_capita = pd.read_excel(raw_path + "gdp.xlsx",
                               sheet_name = "GDPpc",
                               header = [0,1,2],
                               index_col = 0)
maddison_population = pd.read_excel(raw_path + "gdp.xlsx",
                                    sheet_name = "Population",
                                    header = [0,1,2],
                                    index_col = 0)
gdp = gdp_per_capita * maddison_population.values * 1e3 # compute GDP from GDP per capita and population in thousands
gdp = gdp.loc[data_range, :] # select desired years
gdp.index = gdp.index.astype(int)
gdp.columns = gdp.columns.get_level_values(2)
gdp.columns.name = ""
gdp.index.name = None
gdp = gdp_scaling * gdp
gdp.to_csv(processed_path + "gdp.csv")

#### Energy

Raw data units are terajoules

In [25]:
world_energy_balances = pd.read_csv(raw_path + "energy.txt",
                                    sep = r"\s+",
                                    header = None)
energy_classes = ["TOTAL",
				  				"ELECTR",
									"HEAT",
									"COAL",
									"PEAT",
									"OILSHALE",
									"CRNGFEED",
									"TOTPRODS",
									"NATGAS",
									"COMRENEW"]
industry = ["TFC"] # total final consumption
units = ["TJ"] # energy units are in terajoules
energy = world_energy_balances[world_energy_balances[1].isin(energy_classes)]
energy = energy[energy[3].isin(industry)]
energy = energy[energy[4].isin(units)]
energy = energy.drop(3, axis=1)
energy = energy.drop(4, axis=1)
energy[5] = pd.to_numeric(energy[5], errors="coerce")
energy[6] = pd.MultiIndex.from_arrays([energy[0], energy[1]])
energy = energy.pivot_table(index = 2, columns = 6, values = 5)
energy.columns = pd.MultiIndex.from_tuples(energy.columns)
energy = energy.loc[data_range, :] # select desired years
energy.index.name = None

# some of the possible columns are not represented which causes issues later on
# so we add these columns back in with NaNs to avoid slicing errors
energy_regions = energy.columns.get_level_values(0).unique()
energy_types = energy.columns.get_level_values(1).unique()
tuple_list = []
for region in energy_regions:
	for type_ in energy_types:
		if (region, type_) not in energy.columns:
			tuple_list.append((region, type_))
df_temp = pd.DataFrame(nan,
					   				   index=energy.index,
											 columns=tuple_list)
energy = pd.concat([energy, df_temp],
				           axis=1)
# energy[tuple_list] = nan
energy = energy.T.sort_index().T
energy = energy_scaling * energy
energy.to_csv(processed_path + "energy.csv")

#### Emissions

Raw data units are gigagrams

In [26]:
emissions = pd.read_excel(raw_path + "emissions.xlsx",
                          sheet_name = "IPCC 2006",
                          skiprows = 9,
                          index_col = [0,1,2,3,4,5,6,7],
                          header= 0)
emissions = emissions.T
index = [int(idx[2:]) for idx in emissions.index] # take off "Y_" from raw data index
emissions.index = index
emissions = emissions.loc[data_range_emissions, :]
emissions = emissions_scaling * emissions

# combustion emissions from electricity and heat production
cols_elec_n_heat = emissions.columns.get_level_values(4).isin(["1.A.1.a"])
emissions_elec_n_heat = emissions.loc[:, cols_elec_n_heat] # select electricity and heat combustion emissions
emissions_elec_n_heat.columns = emissions_elec_n_heat.columns.get_level_values(2)
emissions_elec_n_heat.to_csv(processed_path + "emissions_elec_n_heat.csv")

# combustion emissions from direct fuel use
categories = emissions.columns.get_level_values(4).values
combustion_mask = [("1.A" in category) for category in categories] # select combustion emissions categories
nonelectric_mask = ~(categories == "1.A.1.a")
fuel_mask = nonelectric_mask & combustion_mask 
emissions_direct = emissions.loc[:, fuel_mask] # select non-electric and non-heat combustion emissions categories
emissions_direct = emissions_direct.T.groupby(level=2).sum()
emissions_direct = emissions_direct.T
emissions_direct.to_csv(processed_path + "emissions_direct.csv")

## Make Time Series

### Loading saved data back in if you didn't just process and save it

In [2]:
energy = pd.read_csv(processed_path + "energy.csv",
                     index_col=0,
                     header=[0,1])
energy_regions = energy.columns.get_level_values(0).unique()
energy_types = energy.columns.get_level_values(1).unique()

population = pd.read_csv(processed_path + "population.csv",
												 index_col=0,
                         header=0)
population_regions = energy.columns.get_level_values(0).unique()

gdp = pd.read_csv(processed_path + "gdp.csv",
									index_col=0,
                  header=0)
gdp_regions = energy.columns.get_level_values(0).unique()

emissions_elec_n_heat = pd.read_csv(processed_path + "emissions_elec_n_heat.csv",
																						index_col=0,
                                            header=0)
emissions_elec_n_heat_regions = energy.columns.get_level_values(0).unique()

emissions_direct = pd.read_csv(processed_path + "emissions_direct.csv",
																		 index_col=0,
                                     header=0)
emissions_direct_regions = energy.columns.get_level_values(0).unique()

### Regrouping the data into the desired regions

In [3]:
def group_energy(df: pd.core.frame.DataFrame,
				   			 energy_types: list,
								 new_regions: list,
								 region_aggregations: dict) -> pd.core.frame.DataFrame:
	"""
	Group World Energy Balance data into the desired regions

	Inputs:
		df:
			the dataframe to be compacted
		categories:
			energy categories in the dataframe
		new_regions:
			new regions to be added to the dataframe by adding and 
			then subtracting off data
		region_aggregations:
			region definitions

	Returns:
		df_grouped:
			a dataframe grouped into the desired regions
	"""
	categories = list(product(energy_types))

	def tuple_filter(var):
		if isinstance(var, tuple):
			var = var
		else:
			var = (var,)
		return var
		
	df_grouped = pd.DataFrame()
	df_zeroed = df.fillna(0) # fill NaNs with zeros so addition can be done
	for continent_key, inner_dict in region_aggregations.items():
		for region_key, subregions in inner_dict.items():
			flag = True	
			for region in subregions:
				if flag:
					temp = df_zeroed[region]
					flag = False
				else:
					temp = temp + df_zeroed[region].values
			columns = pd.MultiIndex.from_tuples([(continent_key, region_key,) + tuple_filter(el) for el in temp.columns])
			temp.columns = columns
			df_grouped = pd.concat([df_grouped, temp],
						                 axis=1)

	for region in new_regions:
		columns = pd.MultiIndex.from_tuples([region + el for el in categories])
		df_grouped[columns] = df_grouped[(region[0], region[1] + " Add")] - \
			                    df_grouped[(region[0], region[1] + " Subtract")].values
		df_grouped = df_grouped.drop([(region[0], region[1] + " Add"), (region[0], region[1] + " Subtract")], axis = 1)
	
	df_grouped = df_grouped.T.sort_index().T

	return df_grouped

In [4]:
def group_data(df: pd.core.frame.DataFrame,
							 new_regions: list,
							 region_aggregations: dict,
							 translator: dict) -> pd.core.frame.DataFrame:
	"""
	Groups dataframe into the desired regions,
	as defined in groupings.py

	Inputs:
		df:
			the dataframe to be compacted
		new_regions:
			new regions to be added to the dataframe by adding and 
			then subtracting off data
		region_aggregations:
			region definitions
		translator:
			translates df naming conventions to World Energy
			Balances data conventions

	Returns:
		df_grouped:
			a dataframe grouped into the desired regions
		"""
	df_grouped = pd.DataFrame()
	for continent_key, inner_dict in region_aggregations.items():
		for region_key, regions in inner_dict.items():
			flag = True	
			for region in regions:
				if flag:
					temp = df[translator[region]].sum(axis=1)
					flag = False
				else:
					temp = temp + df[translator[region]].sum(axis=1).values
			temp = temp.to_frame()
			temp.columns = pd.MultiIndex.from_tuples([(continent_key, region_key)])
			df_grouped = pd.concat([df_grouped,temp],axis=1)

	for region in new_regions:
		df_grouped[region] = df_grouped[(region[0], region[1] + " Add")] - \
								 df_grouped[(region[0], region[1] + " Subtract")].values
		df_grouped = df_grouped.drop([(region[0], region[1] + " Add"), (region[0], region[1] + " Subtract")],
			   				 axis = 1)
		
	df_grouped = df_grouped.T.sort_index().T

	return df_grouped

In [5]:
new_regions = [("Africa", "Other Africa"), ("Asia", "N Asia")]
region_aggregations = gp.region_aggregations

### Loading in grouped data

In [20]:
population_grouped = group_data(population,
												        new_regions,
                                region_aggregations,
												        gp.population_translator)

gdp_grouped = group_data(gdp,
												 new_regions,
                         region_aggregations,
												 gp.gdp_translator)

energy_grouped = group_energy(energy,
												      energy_types,
															new_regions,
															region_aggregations)

emissions_elec_n_heat_grouped = group_data(emissions_elec_n_heat,
												                   new_regions,
                                           region_aggregations,
												                   gp.emissions_translator)

emissions_direct_grouped = group_data(emissions_direct,
												              new_regions,
                                      region_aggregations,
												              gp.emissions_translator)

### Save Population Time Series

In [ ]:
population_save = population_grouped.copy()
population_global = population_save.sum(axis=1)
population_save[("Global", "Global")] = population_global
population_save.to_csv(processed_path + "population_grouped.csv")
# population_save[("Global", "Global")].plot()

### Construct GDP per Capita Time Series

In [19]:
gdp_per_capita = pd.DataFrame()
for region in gdp_grouped.columns.get_level_values(1).unique():
	gdp_region = gdp_grouped.loc[:,idx[:, region]]
	population_region = population_grouped.loc[:,idx[:, region]]
	gdp_per_capita_region = gdp_region.divide(population_region, axis = 1)
	gdp_per_capita = pd.concat([gdp_per_capita, gdp_per_capita_region], axis=1)

# Add global trend
gdp_global = gdp_grouped.sum(axis=1)
population_global = population_grouped.sum(axis=1)
gdp_per_capita[("Global", "Global")] = gdp_global.divide(population_global)
# gdp_per_capita[("Global", "Global")].plot()

# Set zero to NaN
gdp_per_capita = gdp_per_capita.replace(0, nan)

In [27]:
gdp_per_capita.to_csv(processed_path + "gdp_per_capita.csv")

### Construct the Energy per GDP Time Series

In [18]:
energy_per_gdp = pd.DataFrame()
for region in energy_grouped.columns.get_level_values(1).unique():
	temp = energy_grouped.loc[:,idx[:, region, "TOTAL"]].divide(gdp_grouped.loc[:,idx[:, region]].values, axis = 1)
	energy_per_gdp = pd.concat([energy_per_gdp, temp],axis=1)
energy_per_gdp.columns = energy_per_gdp.columns.droplevel(2)
energy_per_gdp = energy_per_gdp.T.sort_index().T # converting to Megajoules per dollar
energy_per_gdp = energy_per_gdp.replace(0, nan) # removing zeros

# Add global trend
total_energy = energy_grouped.loc[:,idx[:, :, "TOTAL"]]
total_energy_global = total_energy.sum(axis=1)
gdp_global = gdp_grouped.sum(axis=1)
energy_per_gdp.loc[data_range_emissions, ("Global", "Global")] = total_energy_global.divide(gdp_global)
# energy_per_gdp[("Global", "Global")].plot()

In [28]:
energy_per_gdp.to_csv(processed_path + "energy_per_gdp.csv")

### Construct the Fraction of Electricity and Heat Time Series

In [16]:
fraction_elec_n_heat = pd.DataFrame()
for region in energy_grouped.columns.get_level_values(1).unique():
	elec_n_heat = energy_grouped.loc[:,idx[:, region, "ELECTR"]] + \
		            energy_grouped.loc[:,idx[:, region, "HEAT"]].values
	divider = energy_grouped.loc[:,idx[:, region, "TOTAL"]]
	fraction_elec_n_heat_region = elec_n_heat.divide(divider.values)
	fraction_elec_n_heat = pd.concat([fraction_elec_n_heat, fraction_elec_n_heat_region],axis=1)
fraction_elec_n_heat.columns = fraction_elec_n_heat.columns.droplevel(2)
fraction_elec_n_heat = fraction_elec_n_heat.T.sort_index().T

# Add global trend
elec_n_heat_energy = energy_grouped.loc[:, idx[:, :, "ELECTR"]] + \
	                   energy_grouped.loc[:, idx[:, :, "HEAT"]].values
elec_n_heat_energy_global = elec_n_heat_energy.sum(axis=1)
total_energy = energy_grouped.loc[:,idx[:, :, "TOTAL"]]
total_energy_global = total_energy.sum(axis=1)
fraction_elec_n_heat[("Global", "Global")] = elec_n_heat_energy_global.divide(total_energy_global)
# fraction_elec_n_heat[("Global", "Global")].plot()

In [29]:
fraction_elec_n_heat.to_csv(processed_path + "fraction_elec_n_heat.csv")

### Emissions per Unit of Electricity and Heat Time Series

In [ ]:
emissions_per_elec_n_heat = pd.DataFrame()
for region in energy_grouped.columns.get_level_values(1).unique():
	elec_n_heat_region = energy_grouped.loc[:,idx[:, region, "ELECTR"]] + \
		                   energy_grouped.loc[:,idx[:, region, "HEAT"]].values
	emissions_per_elec_n_heat_region = emissions_elec_n_heat_grouped.loc[:, idx[:, region]]
	emissions_per_elec_n_heat_region = emissions_per_elec_n_heat_region.divide(elec_n_heat_region.loc[data_range_emissions, :].values, axis = 1)
	emissions_per_elec_n_heat = pd.concat([emissions_per_elec_n_heat, emissions_per_elec_n_heat_region], axis=1)
emissions_per_elec_n_heat = emissions_per_elec_n_heat.T.sort_index().T # grams of CO2 per Megajoule

# Add global trend
emissions_elec_n_heat_global = emissions_elec_n_heat_grouped.sum(axis=1)
elec_n_heat_energy = energy_grouped.loc[:, idx[:, :, "ELECTR"]] + \
	                   energy_grouped.loc[:, idx[:, :, "HEAT"]].values
elec_n_heat_energy_global = elec_n_heat_energy.sum(axis=1)
emissions_per_elec_n_heat[("Global", "Global")] = emissions_elec_n_heat_global.divide(elec_n_heat_energy_global)
# emissions_per_elec_n_heat[("Global", "Global")].plot()

# Set infinity or zero to NaN
emissions_per_elec_n_heat = emissions_per_elec_n_heat.replace(0, nan)
emissions_per_elec_n_heat.replace([np.inf, -np.inf], nan, inplace=True)

In [30]:
emissions_per_elec_n_heat.to_csv(processed_path + "emissions_per_elec_n_heat.csv")

### Emissions per Unit of Other Energy Time Series

In [32]:
emissions_per_direct = pd.DataFrame()
for region in energy_grouped.columns.get_level_values(1).unique():
	total_energy = energy_grouped.loc[:,idx[:, region, "TOTAL"]]
	elec_n_heat_energy = energy_grouped.loc[:, idx[:, region, "ELECTR"]].values + \
		                   energy_grouped.loc[:, idx[:, region, "HEAT"]].values
	direct_energy = total_energy - elec_n_heat_energy
	temp = emissions_direct_grouped.loc[:, idx[:, region]].divide(direct_energy.loc[data_range_emissions, :].values, axis = 1)
	emissions_per_direct = pd.concat([emissions_per_direct, temp],axis=1)
emissions_per_direct = emissions_per_direct.T.sort_index().T

# Add global trend
emissions_direct_global = emissions_direct_grouped.sum(axis=1)
total_energy = energy_grouped.loc[:,idx[:, :, "TOTAL"]]
elec_n_heat_energy = energy_grouped.loc[:, idx[:, :, "ELECTR"]].values + \
	                   energy_grouped.loc[:, idx[:, :, "HEAT"]].values
direct_energy_global = total_energy - elec_n_heat_energy
direct_energy_global = direct_energy_global.sum(axis=1)
emissions_per_direct[("Global", "Global")] = emissions_direct_global.divide(direct_energy_global)
# emissions_per_direct[("Global", "Global")].plot()

# Set infinity or zero to NaN
emissions_per_direct = emissions_per_direct.replace(0, nan)
emissions_per_direct.replace([np.inf, -np.inf], np.nan, inplace=True)

In [33]:
emissions_per_direct.to_csv(processed_path + "emissions_per_direct.csv")

## Plotting

### Load in time series

In [3]:
population_grouped = pd.read_csv(processed_path + "population_grouped.csv",
                                 index_col=0,
                     		         header=[0,1])

gdp_per_capita = pd.read_csv(processed_path + "gdp_per_capita.csv",
                             index_col=0,
                     				 header=[0,1])

energy_per_gdp = pd.read_csv(processed_path + "energy_per_gdp.csv",
                             index_col=0,
                     				 header=[0,1])

fraction_elec_n_heat = pd.read_csv(processed_path + "fraction_elec_n_heat.csv",
                                   index_col=0,
                     				       header=[0,1])

emissions_per_elec_n_heat = pd.read_csv(processed_path + "emissions_per_elec_n_heat.csv",
                                        index_col=0,
                     				            header=[0,1])

emissions_per_direct = pd.read_csv(processed_path + "emissions_per_direct.csv",
                                   index_col=0,
                     				       header=[0,1])

### Plotting Functions

In [16]:
def make_plots(df: pd.core.frame.DataFrame,
				 			 path: str,
				 			 type: str,
							 ylabel: str,
							 scale: str = "linear") -> None:
	"""
	Produce time series plots

	Inputs:
		df:
			dataframe with time series to visualize
		path:
			directory where the visalizations are to be stored
		type:
			time series type
		ylabel:
			plot y label
		scale:
			the y scale for the plot
	"""
	regions = df.columns.get_level_values(0).unique()

	colors = ["firebrick",
		        "mediumvioletred",
						"darkorange",
						"goldenrod",
						"forestgreen",
						"darkolivegreen",
						"mediumblue",
						"darkcyan",
						"indigo",
						"deeppink",
						"saddlebrown",
						"black"]
	
	for region in regions:
		# create individual plot
		label = df[region].columns.get_level_values(0).unique()
		df_region = df.loc[:,idx[region,:]]
		ax = df_region.plot(alpha=0.9,
									      colormap=ListedColormap(colors),
												label=region,
												zorder=1)
		ax.legend(label,
			        loc="upper center",
							bbox_to_anchor=(0.5, -0.15),
							ncol=3)
		
		directory = path + "/" + type + "/individual/"
		os.makedirs(directory,
			          exist_ok = True)
		plt.ylabel(ylabel)
		plt.xlabel("Year")
		plt.title(region)
		plt.yscale(scale)	
		plt.tight_layout()
		plt.savefig(directory + region + ".png",
			          bbox_inches="tight")

		# add in other regions for the comparison plot
		for other_region in regions:
			if region != other_region:
				df.loc[:,idx[other_region,:]].plot(alpha=0.1,
									                         color="gray",
																					 ax=ax,
																					 legend=False,
																					 zorder=0)
		ax.legend(label,
			        loc="upper center",
			        bbox_to_anchor=(0.5, -0.15),
							ncol=3)
		directory = path + "/" + type + "/comparison/"
		os.makedirs(directory,
			          exist_ok = True)
		plt.ylabel(ylabel)
		plt.xlabel("Year")
		plt.title(region)
		plt.yscale(scale)
		plt.tight_layout()
		plt.savefig(directory + region + ".png", bbox_inches="tight")
		plt.close()

In [ ]:
def make_plots_presentation(df,
							              path,
														type,
														regions,
														ylabel,
														scale = "linear"):
	"""
	Produce time series plots

	Inputs:
		df:
			dataframe with time series to visualize
		path:
			directory where the visalizations are to be stored
		regions:
			region indices to be plot
		type:
			time series type
		ylabel:
			plot y label
		scale:
			the y scale for the plot
	"""

	colors = ["black",
	          "brown",
						"gray",
	          "red",
	          "darkorange",
			  	  "gold",
	          "forestgreen",
	          "darkolivegreen",
	          "mediumblue",
						"darkcyan",
						"indigo",
						"deeppink"]

	presentation_df = df[regions]
	
	label = presentation_df.columns.get_level_values(1).unique()
	ax = presentation_df.plot(alpha = 0.7, colormap = ListedColormap(colors), label = regions, zorder = 1)
	ax.legend(label, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
	directory = path + "/" + type + "/presentation/"
	os.makedirs(directory, exist_ok = True)
	plt.ylabel(ylabel)
	plt.xlabel("Year")
	plt.yscale(scale)
	plt.tight_layout()
	plt.savefig(directory + "presentation.png", bbox_inches="tight")
	plt.close()

### Ratio Plots

In [7]:
regions = gdp_per_capita.columns

In [ ]:
## Uncomment to see region numbers
# for i, region in enumerate(gdp_per_capita.columns):
#     print(f"{i}: {region[1]}")

#### Population

In [19]:
make_plots(population_grouped,
           viz_path,
           "population",
           "Number of Individuals",
           scale = "log")

In [ ]:
selection_vec = np.array([0, 4, 5, 7, 8, 12, 14, 16, 18, 19, 20, 22])
presentation_regions = regions[selection_vec]
make_plots_presentation(population_grouped,
           							viz_path,
           							"population",
                        presentation_regions,
           							"Number of Individuals",
           							scale = "log")

#### GDP per Capita

Normal GDP per Capita

In [ ]:
make_plots(gdp_per_capita,
           viz_path,
           "gdp_per_capita",
           "2026 USD per Capita")

In [53]:
selection_vec = np.array([1, 4, 7, 8, 10, 13, 15, 17, 19, 20, 21, 23])
presentation_regions = regions[selection_vec]
make_plots_presentation(gdp_per_capita,
                        viz_path,
                        "gdp_per_capita",
                        presentation_regions,
                        "2026 USD per Capita")

GDP per Capita Ratio

In [29]:
gdp_per_capita_ratio = gdp_per_capita.div(gdp_per_capita[("North America", "USA and Canada")],
                                          axis=0)

In [ ]:
make_plots(gdp_per_capita_ratio,
           viz_path,
           "gdp_per_capita_ratio",
           "Ratio of GDP per Capita between \n Region and USA & Canada")

In [54]:
selection_vec = np.array([1, 4, 7, 8, 10, 13, 15, 17, 19, 20, 21, 23])
presentation_regions = regions[selection_vec]
make_plots_presentation(gdp_per_capita_ratio,
                        viz_path,
                        "gdp_per_capita_ratio",
                        presentation_regions,
                        "Ratio of GDP per Capita between \n Region and USA & Canada")

#### Energy per GDP

In [ ]:
make_plots(energy_per_gdp,
           viz_path,
					 "energy_per_gdp",
					 "Megajoules per 2026 USD",
					 scale = "log")

In [18]:
selection_vec = np.array([16, 18, 20, 5, 22, 4, 0, 13, 8, 6, 11, 14])
presentation_regions = regions[selection_vec]
make_plots_presentation(energy_per_gdp,
           		          viz_path,
					              "energy_per_gdp",
                        presentation_regions,
					              "Megajoules per 2026 USD",
					              scale = "log")

#### Fraction of Energy from Electricity and Heat Plants

In [ ]:
make_plots(fraction_elec_n_heat,
           viz_path,
					 "fraction_elec_n_heat",
           "Fraction of Energy Consumption from\n Electricity and Heat Plants")

In [49]:
selection_vec = np.array([0, 3, 8, 4, 11, 14, 15, 18, 20, 21, 22, 24])
presentation_regions = regions[selection_vec]
make_plots_presentation(fraction_elec_n_heat,
           						  viz_path,
					 						  "fraction_elec_n_heat",
                        presentation_regions,
           						  "Fraction of Energy Consumption from\n Electricity and Heat Plants")

#### Emissions per Unit of Electricity and Heat Plant Output

In [161]:
make_plots(emissions_per_elec_n_heat,
           viz_path,
           "emissions_per_elec_n_heat",
           "Grams of CO$_2$ per Megajoule of\n Electricity and Heat Plant Output")

In [50]:
selection_vec = np.array([1, 4, 9, 5, 12, 13, 16, 19, 21, 6, 23, 0])
presentation_regions = regions[selection_vec]
make_plots_presentation(emissions_per_elec_n_heat,
                        viz_path,
                        "emissions_per_elec_n_heat",
                        presentation_regions,
                        "Grams of CO$_2$ per Megajoule of\n Electricity and Heat Plant Output")

#### Emissions per Other Energy

In [163]:
make_plots(emissions_per_direct,
           viz_path,
           "emissions_per_direct",
           "Grams of CO$_2$ per Megajoule of\n Direct Fuel Use")

In [51]:
selection_vec = np.array([2, 5, 10, 6, 13, 14, 15, 20, 22, 7, 24, 1])
presentation_regions = regions[selection_vec]
make_plots_presentation(emissions_per_direct,
                        viz_path,
                        "emissions_per_direct",
                        presentation_regions,
                        "Grams of CO$_2$ per Megajoule of\n Direct Fuel Use")